In [0]:
from pyspark.sql import functions as F

# ============================================================
# CONFIGURATION
# ============================================================

CATALOG = "workspace"

SILVER_SCHEMA = "firmable_v2_silver"
BRONZE_SCHEMA = "firmable_v2_bronze"
GOLD_SCHEMA = "firmable_v2_gold"

HIERARCHY_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.employee_hierarchy"
COMPANY_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.companies"
VALIDATION_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.hierarchy_validation_summary"

GOLD_COMPANIES = f"{CATALOG}.{GOLD_SCHEMA}.gold_companies"
GOLD_EMPLOYEES = f"{CATALOG}.{GOLD_SCHEMA}.gold_employee_hierarchy"
GOLD_EDGES = f"{CATALOG}.{GOLD_SCHEMA}.gold_org_chart_edges"
GOLD_METRICS = f"{CATALOG}.{GOLD_SCHEMA}.gold_company_metrics"

# ============================================================
# 1. LOAD SOURCE TABLES
# ============================================================

hierarchy = spark.table(HIERARCHY_TABLE)
companies = spark.table(COMPANY_TABLE)
validation = spark.table(VALIDATION_TABLE)

# Do not publish if structural validation failed.
failed_checks = validation.filter(
    F.col("passed") == False
).count()

assert failed_checks == 0, (
    f"Gold publication stopped: {failed_checks} hierarchy checks failed."
)

# ============================================================
# 2. PREPARE COMPANY DIRECTORY
# ============================================================

# Retain available company metadata without assuming
# every optional source column exists.

preferred_company_columns = [
    "company_id",
    "company_name",
    "industry",
    "org_structure",
    "company_size_category",
    "employee_count"
]

available_company_columns = [
    column for column in preferred_company_columns
    if column in companies.columns
]

assert "company_id" in available_company_columns

company_directory = companies.select(
    *available_company_columns
).dropDuplicates(["company_id"])

# ============================================================
# 3. PREPARE EMPLOYEE HIERARCHY
# ============================================================

employee_data = hierarchy.alias("employee")

manager_data = hierarchy.select(
    "company_id",
    F.col("employee_id").alias("lookup_manager_id"),
    F.col("company_name").alias("manager_company_name"),
    F.col("job_title_clean").alias("manager_job_title"),
    F.col("department").alias("manager_department"),
    F.col("business_unit").alias("manager_business_unit"),
    F.col("team_name").alias("manager_team_name"),
    F.col("inferred_seniority_level").alias("manager_seniority_level"),
    F.col("management_authority").alias("manager_authority")
).alias("manager")

gold_employees = (
    employee_data
    .join(
        manager_data,
        (
            F.col("employee.company_id") ==
            F.col("manager.company_id")
        ) & (
            F.col("employee.manager_id") ==
            F.col("manager.lookup_manager_id")
        ),
        "left"
    )
    .select(
        "employee.*",
        F.col("manager.manager_job_title"),
        F.col("manager.manager_department"),
        F.col("manager.manager_business_unit"),
        F.col("manager.manager_team_name"),
        F.col("manager.manager_seniority_level"),
        F.col("manager.manager_authority")
    )
    .withColumn(
        "inference_method",
        F.lit("RULE_BASED")
    )
    .withColumn(
        "confidence_score_type",
        F.lit("RULE_BASED_SCORE_NOT_PROBABILITY")
    )
    .withColumn(
        "publication_version",
        F.lit("2.0")
    )
)

# ============================================================
# 4. PREPARE ORG CHART EDGES
# ============================================================

gold_edges = gold_employees.filter(
    F.col("manager_id").isNotNull()
).select(
    "company_id",
    "manager_id",
    "employee_id",
    "job_title_clean",
    "department",
    "candidate_score",
    "relationship_type",
    "assignment_round",
    "root_employee_id",
    "hierarchy_depth",
    "assignment_status"
).withColumn(
    "inference_method",
    F.lit("RULE_BASED")
)

# ============================================================
# 5. COMPANY-LEVEL METRICS
# ============================================================

company_metrics = gold_employees.groupBy(
    "company_id"
).agg(
    F.count("*").alias("total_employees"),

    F.sum(
        F.when(
            F.col("manager_id").isNotNull(), 1
        ).otherwise(0)
    ).alias("assigned_employees"),

    F.sum(
        F.when(
            F.col("is_root"), 1
        ).otherwise(0)
    ).alias("component_roots"),

    F.sum(
        F.when(
            F.col("root_type") == "CEO_ROOT", 1
        ).otherwise(0)
    ).alias("ceo_roots"),

    F.sum(
        F.when(
            F.col("root_type") == "UNASSIGNED_ROOT", 1
        ).otherwise(0)
    ).alias("unassigned_roots"),

    F.max("hierarchy_depth").alias("maximum_hierarchy_depth"),

    F.avg(
        F.when(
            F.col("manager_id").isNotNull(),
            F.col("candidate_score")
        )
    ).alias("average_assignment_score"),

    F.min(
        F.when(
            F.col("manager_id").isNotNull(),
            F.col("candidate_score")
        )
    ).alias("minimum_assignment_score")
).withColumn(
    "assignment_coverage",
    F.col("assigned_employees") / F.col("total_employees")
).withColumn(
    "unassigned_employee_count",
    F.col("total_employees") - F.col("assigned_employees")
)

# ============================================================
# 6. BUILD COMPANY DIRECTORY WITH METRICS
# ============================================================

gold_companies = company_directory.join(
    company_metrics,
    ["company_id"],
    "left"
).withColumn(
    "inference_method",
    F.lit("RULE_BASED")
).withColumn(
    "publication_version",
    F.lit("2.0")
)

# ============================================================
# 7. PUBLISH GOLD TABLES
# ============================================================

def publish_delta(df, table_name):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

publish_delta(gold_companies, GOLD_COMPANIES)
publish_delta(gold_employees, GOLD_EMPLOYEES)
publish_delta(gold_edges, GOLD_EDGES)
publish_delta(company_metrics, GOLD_METRICS)

# ============================================================
# 8. PUBLICATION VALIDATION
# ============================================================

published_companies = spark.table(GOLD_COMPANIES)
published_employees = spark.table(GOLD_EMPLOYEES)
published_edges = spark.table(GOLD_EDGES)
published_metrics = spark.table(GOLD_METRICS)

company_count = published_companies.count()
employee_count = published_employees.count()
edge_count = published_edges.count()
metrics_count = published_metrics.count()

duplicate_employees = (
    published_employees
    .groupBy("company_id", "employee_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicate_edges = (
    published_edges
    .groupBy("company_id", "employee_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

missing_manager_details = published_employees.filter(
    F.col("manager_id").isNotNull() &
    F.col("manager_job_title").isNull()
).count()

invalid_coverage = published_metrics.filter(
    (F.col("assignment_coverage") < 0) |
    (F.col("assignment_coverage") > 1)
).count()

assert employee_count == 89576
assert duplicate_employees == 0
assert duplicate_edges == 0
assert missing_manager_details == 0
assert invalid_coverage == 0
assert edge_count == published_employees.filter(
    F.col("manager_id").isNotNull()
).count()

# ============================================================
# 9. PUBLICATION SUMMARY
# ============================================================

print("=" * 65)
print("GOLD LAYER PUBLICATION COMPLETED")
print("=" * 65)

print("Companies published:", company_count)
print("Employees published:", employee_count)
print("Org chart edges published:", edge_count)
print("Company metric records:", metrics_count)

print("\nPUBLICATION VALIDATION")
print("Duplicate employee records:", duplicate_employees)
print("Duplicate reporting edges:", duplicate_edges)
print("Missing manager details:", missing_manager_details)
print("Invalid coverage records:", invalid_coverage)

print("\nCOMPANY COVERAGE SUMMARY")
published_metrics.select(
    F.count("*").alias("companies"),
    F.avg("assignment_coverage").alias("average_company_coverage"),
    F.min("assignment_coverage").alias("minimum_company_coverage"),
    F.max("assignment_coverage").alias("maximum_company_coverage")
).show(truncate=False)

print("\nGOLD TABLES")
print(GOLD_COMPANIES)
print(GOLD_EMPLOYEES)
print(GOLD_EDGES)
print(GOLD_METRICS)

print("\nALL GOLD PUBLICATION CHECKS PASSED")

In [0]:
from pyspark.sql import functions as F

# ============================================
# CONFIGURATION
# ============================================

VOLUME_NAME = "workspace.firmable_v2_gold.firmable_exports"

EXPORT_PATH = "/Volumes/workspace/firmable_v2_gold/firmable_exports"

TABLES = {
    "gold_companies": "workspace.firmable_v2_gold.gold_companies",
    "gold_employee_hierarchy": "workspace.firmable_v2_gold.gold_employee_hierarchy",
    "gold_org_chart_edges": "workspace.firmable_v2_gold.gold_org_chart_edges",
    "gold_company_metrics": "workspace.firmable_v2_gold.gold_company_metrics"
}

# ============================================
# CREATE VOLUME
# ============================================

spark.sql(f"CREATE VOLUME IF NOT EXISTS {VOLUME_NAME}")

print("Export volume ready:", VOLUME_NAME)

# ============================================
# EXPORT GOLD TABLES
# ============================================

export_results = []

for file_name, table_name in TABLES.items():

    print(f"\nExporting: {table_name}")

    df = spark.table(table_name)
    row_count = df.count()

    temp_path = f"{EXPORT_PATH}/_{file_name}_temp"
    final_path = f"{EXPORT_PATH}/{file_name}.csv.gz"

    # Remove previous exports if they exist.
    dbutils.fs.rm(temp_path, True)
    dbutils.fs.rm(final_path, True)

    # Write a single compressed CSV file.
    (
        df.coalesce(1)
        .write
        .mode("overwrite")
        .option("header", "true")
        .option("compression", "gzip")
        .csv(temp_path)
    )

    # Find the generated CSV part file.
    temp_files = dbutils.fs.ls(temp_path)

    part_files = [
        f for f in temp_files
        if f.name.startswith("part-")
        and f.name.endswith(".csv.gz")
    ]

    if len(part_files) != 1:
        raise RuntimeError(
            f"{file_name}: Expected exactly one CSV part file, "
            f"but found {len(part_files)}. "
            f"Files: {[f.name for f in temp_files]}"
        )

    # Move the generated file to its final filename.
    move_success = dbutils.fs.mv(
        part_files[0].path,
        final_path
    )

    if not move_success:
        raise RuntimeError(f"Could not move exported file for {file_name}")

    # Remove temporary directory.
    dbutils.fs.rm(temp_path, True)

    # Verify using filename, not full path equality.
    final_files = dbutils.fs.ls(EXPORT_PATH)

    matching_files = [
        f for f in final_files
        if f.name.rstrip("/") == f"{file_name}.csv.gz"
    ]

    if len(matching_files) != 1:
        raise RuntimeError(
            f"{file_name}: Export file verification failed. "
            f"Found: {[f.name for f in final_files]}"
        )

    file_size_mb = matching_files[0].size / (1024 * 1024)

    export_results.append(
        (file_name, row_count, round(file_size_mb, 2), "SUCCESS")
    )

    print(f"Rows exported: {row_count:,}")
    print(f"File size: {file_size_mb:.2f} MB")
    print("Status: SUCCESS")

# ============================================
# FINAL EXPORT REPORT
# ============================================

report = spark.createDataFrame(
    export_results,
    ["file_name", "row_count", "size_mb", "status"]
)

print("\n========== GOLD DATA EXPORT SUMMARY ==========")
report.show(truncate=False)

if len(export_results) != len(TABLES):
    raise RuntimeError("Not all Gold tables were exported.")

if any(result[3] != "SUCCESS" for result in export_results):
    raise RuntimeError("One or more Gold table exports failed.")

print("ALL FOUR GOLD TABLES EXPORTED SUCCESSFULLY")
print("EXPORT LOCATION:", EXPORT_PATH)